<a href="https://colab.research.google.com/github/CarlosSantos8/Machine-Learning/blob/main/proyectofinal_fase2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Librerías

In [1]:
!pip install keras_tuner
!pip install deep_translator
!pip install langdetect

In [2]:
import Funciones_PNL as f
import numpy as np
import pandas as pd
import importlib
from sklearn.model_selection import train_test_split
import numpy as np
import pandas as pd  # Cambiado a pd, que es el estándar para manipulación de datos
import random
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import tensorflow as tf # Corregido de 'ft' a 'tf'

# Herramientas para procesar texto (NLP)
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Arquitectura del modelo
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Dense, SimpleRNN, LSTM, Input

# Optimizador (aquí es donde puedes aplicar el Gradient Clipping que vimos)
from tensorflow.keras.optimizers import Adam

from sklearn.preprocessing import LabelEncoder
import joblib
import pickle
np.random.seed(42)
tf.random.set_seed(42)
import json
from sklearn.utils import class_weight

## Carga dataset

In [3]:
# ============================================================
# CARGA DEL DATASET
# ============================================================

# Invoca la función polimórfica 'load_dataset' pasándole la ruta construida.
# La función validará la existencia del archivo, detectará la extensión '.xlsx'
# y utilizará internamente 'pd.read_excel()' para mapear las hojas a un DataFrame.
df = f.load_dataset("ProyectoFinal_Clasificacion_fase12_limpieza_final.parquet")


Cargando dataset...
Archivo: ProyectoFinal_Clasificacion_fase12_limpieza_final.parquet


In [4]:
df.columns

Index(['texto', 'categoria', 'text_colum_clean', 'text_colum_clean_es',
       'text_colum_clean_es_v2'],
      dtype='object')

In [5]:
# 3. Evitar que el ancho de columna trunque el contenido (útil para textos largos)
pd.set_option('display.max_colwidth', None)
# Filtra el DataFrame original manteniendo únicamente las dos columnas especificadas y sobrescribe la variable 'df'.
df = df[['text_colum_clean_es_v2', 'categoria']]
# Renombra las columnas del DataFrame mapeando los nombres antiguos a los nuevos mediante un diccionario y sobrescribe la variable 'df'.
df = df.rename(columns={'text_colum_clean_es_v2': 'texto'})
df.head(20)

,texto,categoria
0,buenas ofertas,Experiencia Positiva
2,el sitio es muy fácil de usar y el servicio muy bueno,Experiencia Positiva
3,es una tienda que tiene buenas promociones y el personal es muy amable y atento,Experiencia Positiva
4,en algunos anaqueles de galletas están vacías y poco producto área de frutas están mezcladas manzanas con diferente código me pasó con las manzanas rojas,"Disponibilidad, Abasto y Surtido"
5,tiene todo lo necesario y mas la atención es muy buena serviciales,Experiencia Positiva
6,todo muy limpio y ordenado,Experiencia Positiva
7,muy bonita sucursal y bien surtida,Experiencia Positiva
8,buen super tienen producto de calidad y variado incluso la conveniencia de pagar con ustedes el estacionamiento con descuento,Experiencia Positiva
9,todo excelente,Experiencia Positiva
10,tienda donde encuentras de todo siempre muy limpio,Experiencia Positiva


In [6]:
df_rnn = df.copy()

In [7]:
df_rnn.head(5)

,texto,categoria
0,buenas ofertas,Experiencia Positiva
2,el sitio es muy fácil de usar y el servicio muy bueno,Experiencia Positiva
3,es una tienda que tiene buenas promociones y el personal es muy amable y atento,Experiencia Positiva
4,en algunos anaqueles de galletas están vacías y poco producto área de frutas están mezcladas manzanas con diferente código me pasó con las manzanas rojas,"Disponibilidad, Abasto y Surtido"
5,tiene todo lo necesario y mas la atención es muy buena serviciales,Experiencia Positiva


# Modelos RNN, LSTM y GRU

## Preprocesamiento

### Limpieza

In [8]:
# Importación de NLTK para tareas de procesamiento de lenguaje natural
import nltk

# Descarga de recursos necesarios para el preprocesamiento:
# 'punkt_tab': Datos para la tokenización de oraciones/palabras
# 'stopwords': Lista de palabras vacías (stop words) para filtrar ruido en el texto
nltk.download('punkt_tab')
nltk.download('stopwords')

# Aplicación de la función de limpieza personalizada definida en el módulo 'f' (Funciones_PNL).
# Se procesa la columna 'texto' para normalizar el corpus; 'verbose=0' se utiliza
# para suprimir la salida de registros durante la ejecución iterativa.
df_rnn['texto_rnn'] = df_rnn['texto'].apply(
    lambda x: f.limpiar_para_rnn(x, verbose=0)
)

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [9]:
df_rnn.head(5)

,texto,categoria,texto_rnn
0,buenas ofertas,Experiencia Positiva,buenas ofertas
2,el sitio es muy fácil de usar y el servicio muy bueno,Experiencia Positiva,sitio muy fácil usar servicio muy bueno
3,es una tienda que tiene buenas promociones y el personal es muy amable y atento,Experiencia Positiva,tienda tiene buenas promociones personal muy amable atento
4,en algunos anaqueles de galletas están vacías y poco producto área de frutas están mezcladas manzanas con diferente código me pasó con las manzanas rojas,"Disponibilidad, Abasto y Surtido",anaqueles galletas vacías poco producto área frutas mezcladas manzanas diferente código pasó manzanas rojas
5,tiene todo lo necesario y mas la atención es muy buena serviciales,Experiencia Positiva,tiene todo necesario mas atención muy buena serviciales


In [10]:
df.columns

Index(['texto', 'categoria'], dtype='object')

### Eliminar columnas innecesarias

In [11]:
# Eliminar columnas que no usarás para el entrenamiento antes del split
df_rnn = df_rnn.drop(columns=['texto'])

In [12]:
df_rnn.head(5)

,categoria,texto_rnn
0,Experiencia Positiva,buenas ofertas
2,Experiencia Positiva,sitio muy fácil usar servicio muy bueno
3,Experiencia Positiva,tienda tiene buenas promociones personal muy amable atento
4,"Disponibilidad, Abasto y Surtido",anaqueles galletas vacías poco producto área frutas mezcladas manzanas diferente código pasó manzanas rojas
5,Experiencia Positiva,tiene todo necesario mas atención muy buena serviciales


### Textos vacíos

Aplicamos la función de limpieza centralizada definida en **'Funciones_PNL'** para filtrar  y eliminar los registros vacíos o nulos de la columna especificada ('texto_rnn').
Esto asegura que el dataset que alimentará la red neuronal solo contenga secuencias con contenido léxico válido, evitando errores en la tokenización y el padding.

In [13]:
df_rnn = f.limpiar_columna_vacios(df_rnn, 'texto_rnn')

Filas detectadas como vacías: 7
                         categoria texto_rnn
2017          Experiencia Positiva          
7350           Ruido / Descartados          
10058          Ruido / Descartados          
15147          Ruido / Descartados          
40024  Operación LCC / Corporativo          


Volvemos a ejercutar, para verificar que ya no hay valores nulos.

In [14]:
df_rnn = f.limpiar_columna_vacios(df_rnn, 'texto_rnn')

Filas detectadas como vacías: 0
Empty DataFrame
Columns: [categoria, texto_rnn]
Index: []


### Duplicados

Identificamos registros duplicados basándonos en la combinación del texto procesado y su categoría asignada. Esto es vital en NLP para evitar que el modelo de red neuronal memorice patrones sobre-representados, lo que podría  causar sobreajuste (overfitting).

In [15]:
df_rnnduplicados = f.obtener_duplicados(df_rnn, ['texto_rnn', 'categoria'])
df_rnnduplicados.head(10)

[*] Modo verbose: Evaluando duplicados en las columnas: ['texto_rnn', 'categoria']
[*] Se encontraron 2545 filas duplicadas.
[*] Ordenando los resultados...


,categoria,texto_rnn
12935,Servicio y Atención (Personal),abierta sucursal
12978,Servicio y Atención (Personal),abierta sucursal
25364,Experiencia Positiva,agradable
37669,Experiencia Positiva,agradable
30147,Experiencia Positiva,agradable lugar
30931,Experiencia Positiva,agradable lugar
34823,Experiencia Positiva,agradable tienda
47892,Experiencia Positiva,agradable tienda
12115,Servicio y Atención (Personal),agregar pedido
14492,Servicio y Atención (Personal),agregar pedido


In [16]:
# Eliminar filas donde el par de columnas [columna_1, columna_2] sea idéntico
df_rnn.drop_duplicates(subset=['texto_rnn','categoria'], keep='first', inplace=True)

In [17]:
df_rnnduplicados = f.obtener_duplicados(df_rnn,['texto_rnn','categoria'])
df_rnnduplicados.head(40)

[*] Modo verbose: Evaluando duplicados en las columnas: ['texto_rnn', 'categoria']
[*] Se encontraron 0 filas duplicadas.
[*] No se encontraron duplicados.


,categoria,texto_rnn


### Inconsistencia

Identificamos registros donde un mismo 'texto_rnn' (la entrada normalizada)  tiene asignadas categorías distintas, lo cual constituye una inconsistencia  en la etiqueta (etiquetado ambiguo). Esta función es crucial para la integridad del entrenamiento de la RNN, ya que  etiquetas contradictorias para el mismo input degradan significativamente la capacidad predictiva del modelo.

In [18]:
df_rnn_inconsistencia = f.obtener_conflictos_categoria(df_rnn, col_texto='texto_rnn', col_cat='categoria')
df_rnn_inconsistencia.head(10) # Inspeccionamos los primeros 10 conflictos

[*] Analizando inconsistencias entre 'texto_rnn' y 'categoria'...
[*] Se encontraron 78 textos con múltiples categorías.
[*] Total de filas afectadas: 159


,categoria,texto_rnn
30930,Operación LCC / Corporativo,atención calidad productos
29694,Experiencia Positiva,atención calidad productos
20005,Experiencia Positiva,atención cliente
22183,Servicio y Atención (Personal),atención cliente
22234,Servicio y Atención (Personal),bajen precios
7016,"Comercial, Precios y Ofertas",bajen precios
14885,Experiencia Positiva,calidad fruta
2018,Operación LCC / Corporativo,calidad fruta
11055,Experiencia Positiva,calidad frutas verduras
43907,Operación LCC / Corporativo,calidad frutas verduras


In [19]:
len(df_rnn)

52854

In [20]:
import importlib
importlib.reload(f) # Esto obliga a Python a leer el archivo de nuevo y actualizar las funciones
df_rnn = f.eliminar_inconsistencias(df_rnn, col_texto='texto_rnn', col_cat='categoria')

[*] Tamaño del DataFrame: 52854
[*] Se eliminaron 78 textos inconsistentes.
[*] Filas eliminadas: 159
[*] Nuevo tamaño del DataFrame: 52695


In [21]:
df_rnn = f.eliminar_inconsistencias(df_rnn, col_texto='texto_rnn', col_cat='categoria')

[*] No se encontraron inconsistencias. Nada que eliminar.


### Depuración de Datos y Manejo de Outliers

Aplicamos una técnica de filtrado estadístico basada en el Rango Intercuartílico (IQR)  y la regla de los bigotes (Tukey's fences) para detectar y eliminar valores atípicos  (outliers) en la longitud de las secuencias de texto. <code> 'factor_extra=1.5' </code> es el multiplicador estándar para definir los límites de los bigotes.
Esto ayuda a que el modelo no se vea afectado por secuencias inusualmente largas  o cortas que podrían desestabilizar el entrenamiento o afectar el desempeño del padding.

In [22]:
importlib.reload(f)
df_rnn = f.filtrar_por_rango_bigotes(df_rnn, 'texto_rnn', factor_extra=1.5)

--- Reporte de Filtrado ---
Filas originales: 52,695
Límite inferior: 3
Límite superior (bigotes): 187
Filas resultantes: 48,287
Filas eliminadas: 4,408

--- 5 Ejemplos ELIMINADOS (del más largo al más corto) ---

[Largo 25786]: i m afraid i wasn t able to deliver the following message this is permanent error i ve given up sorry it didn t work out subject windows qsolicituddereincorporaci fnyaltadelclubrotarior windows qoma distrito to maderasauco yahoocommx below this line is copy of the message i m afraid i wasn t able to deliver the following message this is permanent error i ve given up sorry it didn t work out subject windows qsolicituddereincorporaci fnyaltadelclubrotarior windows qoma distrito to maderasauco yahoocommx below this line is copy of the message i m afraid i wasn t able to deliver the following message this is permanent error i ve given up sorry it didn t work out subject windows qsolicituddereincorporaci fnyaltadelclubrotarior windows qoma distrito to maderasauco ya

In [23]:
len(df_rnn)

48287

### Balanceo de datos

In [24]:
f.analizar_distribucion_clases(df_rnn,'categoria') # Solo pasa el nombre de la columna (categoria)


DISTRIBUCIÓN DE CLASES


,Clase,Frecuencia,Porcentaje
0,Experiencia Positiva,19027,39.40
1,Operación LCC / Corporativo,17984,37.24
2,Servicio y Atención (Personal),8201,16.98
3,"Comercial, Precios y Ofertas",915,1.89
4,"Disponibilidad, Abasto y Surtido",633,1.31
5,Calidad y Merma de Producto,522,1.08
6,"Infraestructura, Higiene y Tienda",441,0.91
7,Operación de Cajas y Pago,308,0.64
8,Ruido / Descartados,129,0.27
9,otros,79,0.16


----------------------------------------------------------------------
Ratio de desbalance: 396.40
ALERTA: Desbalance severo detectado. Considerar técnicas de resampling o class_weights.


## Preparación de datos

### Label Encoder

In [25]:
le = LabelEncoder() # 1. Instanciamos el codificador
df_rnn['categoria'] = le.fit_transform(df_rnn['categoria']) # 2. Aplicamos la transformación directamente sobre la columna original

# 3. Guarda el objeto 'le' para poder revertir los nombres después
# Puedes guardarlo en una variable o incluso como un atributo del df si lo necesitas persistente
joblib.dump(le, 'label_encoder_categoria_rnn.pkl')

print("Columna 'categoria' actualizada con éxito a valores numéricos.")

Columna 'categoria' actualizada con éxito a valores numéricos.


In [26]:
mapeo_inverso = dict(zip(le.transform(le.classes_), le.classes_)) # Esto te da un mapeo claro de {número: nombre}
mapeo_inverso

{np.int64(0): 'Calidad y Merma de Producto',
 np.int64(1): 'Comercial, Precios y Ofertas',
 np.int64(2): 'Disponibilidad, Abasto y Surtido',
 np.int64(3): 'Experiencia Positiva',
 np.int64(4): 'Infraestructura, Higiene y Tienda',
 np.int64(5): 'Omnicanalidad y Postventa',
 np.int64(6): 'Operación LCC / Corporativo',
 np.int64(7): 'Operación de Cajas y Pago',
 np.int64(8): 'Ruido / Descartados',
 np.int64(9): 'Servicio y Atención (Personal)',
 np.int64(10): 'otros'}

### Preparación de input y target

In [27]:
# ============================================================
# 1. PREPARACIÓN DE INPUT (X) Y TARGET (y)
# Mantenemos 'X' e 'y' como Series de Pandas para preservar
# la integridad de los índices y facilitar la división de datos.
# ============================================================

X = df_rnn['texto_rnn'].tolist()  # Serie de Pandas
y = df_rnn['categoria'].tolist()  # Serie de Pandas

In [28]:
longitudes = df_rnn['texto_rnn'].astype(str).str.len() # Creamos una serie con la longitud de cada texto
print(longitudes.describe()) # Mostramos estadísticas básicas

count    48287.000000
mean        59.893387
std         40.048328
min          3.000000
25%         31.000000
50%         47.000000
75%         78.000000
max        187.000000
Name: texto_rnn, dtype: float64


In [29]:
# Ordenar el DataFrame por la longitud del texto de mayor a menor
df_ordenado = df_rnn.assign(longitud=df_rnn['texto_rnn'].astype(str).str.len())
top_largos = df_ordenado.sort_values(by='longitud', ascending=False)
print(top_largos[['texto_rnn', 'longitud']].head(5)) # Mostrar las 5 filas con los textos más largos

                                                                                                                                                                                         texto_rnn  \
8573   acudí sucursal fresko vallarta día ayer olvidé tarjeta banorte caja autopago sucursal fresko vallarta solicito número telefónico hablar sucursal saber si alguien entregó tarjeta guardarla   
22431  no contestan teléfono persona tienda encargada pedido hace preguntas wa después tampoco contesta cobran envío pedido llega bastante retrasado ahora devolver trajeron cosa diferente compré   
1921   todo bien organizado encuentran productos mucha variedad panadería increíble café delicioso deberían colocar más ofertas poco más caros supermercados súper limpio personal amable volvería   
35742  dirijo ustedes expresar inconformidad respecto experiencia vivida día ayer sucursal santa fe visita atención distintos departamentos deficiente particularmente área salchichonería sin emb   
29364  no 

### División de datos

Realizamos la partición del dataset con estratificación
- <code> stratify=y </code>: Asegura que el $70$% de los positivos vayan a train y el $30$% a test (lo mismo para los negativos), manteniendo el balance original del dataset.

In [30]:
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42)
# Verificamos que la proporción se mantuvo (debería haber 50% de cada clase en ambos)
print(f"Entrenamiento: {len(X_train)} muestras")
print(f"Prueba: {len(X_test)} muestras")
print(f"Validacion: {len( y_val)} muestras")



Entrenamiento: 33800 muestras
Prueba: 7244 muestras
Validacion: 7243 muestras


### Tokenización

Instanciamos el Tokenizer

<code> oov_token  </code>: Palabra especial para representar términos que no estén en el vocabulario durante el test

In [31]:
tokenizer = Tokenizer( oov_token="<OOV>")

Ajustamos el tokenizador SOLAMENTE con los datos de entrenamiento. Esto construye el diccionario de palabras basándose en <code> X_train </code>

In [32]:
tokenizer.fit_on_texts(X_train)

Guardamos el tokenizer

In [33]:
with open('tokenizer_rnn.pkl', 'wb') as handle:
    pickle.dump(tokenizer, handle, protocol=pickle.HIGHEST_PROTOCOL)

In [34]:
# Extraemos el diccionario de mapeo generado por el Tokenizer
# La llave es la palabra y el valor es su índice numérico
word_index = tokenizer.word_index

# Mostramos el tamaño real del vocabulario encontrado y un ejemplo
print(f"Total de palabras únicas encontradas: {len(word_index)}")
print("Ejemplo del índice para '<OOV>':", word_index.get("<OOV>"))

Total de palabras únicas encontradas: 18662
Ejemplo del índice para '<OOV>': 1


In [35]:
# Encabezados de la tabla
print(f"{'Índice':<10} | {'Palabra':<20}")
print("-" * 35)

# Recorremos el word_index y lo imprimimos con formato de tabla
# Usamos < para alinear a la izquierda y el número para el ancho de la columna
for palabra, indice in list(word_index.items())[:25]:
    print(f"{indice:<10} | {palabra:<20}")

print("-" * 35)

Índice     | Palabra             
-----------------------------------
1          | <OOV>               
2          | no                  
3          | muy                 
4          | pedido              
5          | productos           
6          | todo                
7          | servicio            
8          | si                  
9          | tienda              
10         | calidad             
11         | siempre             
12         | excelente           
13         | quiero              
14         | bien                
15         | atención            
16         | buena               
17         | buen                
18         | gracias             
19         | tienen              
20         | saber               
21         | entrega             
22         | personal            
23         | más                 
24         | necesito            
25         | favor               
-----------------------------------


### Secuencias Padding

In [36]:
# 1. Convertir textos a secuencias (usando el mismo tokenizer para todo)
X_train_seq = tokenizer.texts_to_sequences(X_train)
X_val_seq = tokenizer.texts_to_sequences(X_val)
X_test_seq = tokenizer.texts_to_sequences(X_test)

In [37]:
for i in range(5):
  print(f"Ejemplo {i}")
  print("-" * 35)
  print(f"Frase original (texto): {X_train[i]}...")
  print(f"Frase convertida (números): {X_train_seq[i]}...")

Ejemplo 0
-----------------------------------
Frase original (texto): gran variedad productos precios razonables...
Frase convertida (números): [105, 30, 5, 27, 1483]...
Ejemplo 1
-----------------------------------
Frase original (texto): faltaron dos unidades cloro...
Frase convertida (números): [190, 66, 2693, 1927]...
Ejemplo 2
-----------------------------------
Frase original (texto): no pude facturar...
Frase convertida (números): [2, 293, 263]...
Ejemplo 3
-----------------------------------
Frase original (texto): quedaron formalmente entregar día ayer tarde spray suavizanteante telas suavitel no suavizante normal no trajeron spray...
Frase convertida (números): [504, 6341, 116, 46, 112, 113, 3472, 8716, 5138, 2118, 2, 2230, 726, 2, 96, 3472]...
Ejemplo 4
-----------------------------------
Frase original (texto): buen surtido productos muy buena calidad...
Frase convertida (números): [17, 50, 5, 3, 16, 10]...


### Longitud Máxima - Entrenamiento

In [38]:
# Calculamos la longitud máxima observada en cada conjunto
# La list comprehension dentro de max() recorre cada secuencia y mide su len()
max_train = max([len(x) for x in X_train_seq])
max_test = max([len(x) for x in X_test_seq])

# Imprimimos los resultados para definir nuestro maxlen del padding
print(f"Longitud máxima en entrenamiento: {max_train}")
print(f"Longitud máxima en prueba: {max_test}")

Longitud máxima en entrenamiento: 36
Longitud máxima en prueba: 31


In [39]:
max_len =max_train  #Se ocupa para evitar fuga de datos

### Padding

Aplicamos 'pad_sequences' para estandarizar todas las secuencias a una longitud fija (**maxlen**). Esto es necesario porque las capas de las redes neuronales (RNN, LSTM, etc.) requieren que todos los inputs dentro de un lote (batch) tengan dimensiones idénticas.
<code> 'padding='post' </code> rellena con ceros al final de la secuencia si el texto es más corto  que 'max_len', asegurando que la estructura de la entrada sea una matriz rectangular.

In [40]:
X_train_pad = pad_sequences(X_train_seq, maxlen=max_len, padding='post')
X_test_pad = pad_sequences(X_test_seq, maxlen=max_len, padding='post')
X_val_pad = pad_sequences(X_val_seq, maxlen=max_len, padding='post')

In [41]:
# Imprimimos las dimensiones de los tensores resultantes
print(f"Tamaño de X_train_pad: {X_train_pad.shape}")
print(f"Tamaño de X_test_pad: {X_test_pad.shape}")

# También podemos ver el total de elementos (celdas) en la matriz de entrenamiento
print(f"Total de datos en la matriz de entrenamiento: {X_train_pad.size}")

Tamaño de X_train_pad: (33800, 36)
Tamaño de X_test_pad: (7244, 36)
Total de datos en la matriz de entrenamiento: 1216800


In [42]:
# Elegimos un índice para comparar (el primero)
i = 0

print("--- TRANSFORMACIÓN DE UN EJEMPLO ---")

# 1. Frase Original (Texto)
print(f"\n1. FRASE ORIGINAL (X_train):")
print(X_train[i])

# 2. Frase Tokenizada (Lista de números de longitud variable)
print(f"\n2. FRASE TOKENIZADA (X_train_seq):")
print(X_train_seq[i])
print(f"Longitud original: {len(X_train_seq[i])} palabras")

# 3. Frase con Padding (Matriz de longitud fija para la red)
print(f"\n3. FRASE CON PADDING (X_train_pad):")
print(X_train_pad[i])
print(f"Longitud final: {len(X_train_pad[i])} (rellenada con ceros)")

--- TRANSFORMACIÓN DE UN EJEMPLO ---

1. FRASE ORIGINAL (X_train):
gran variedad productos precios razonables

2. FRASE TOKENIZADA (X_train_seq):
[105, 30, 5, 27, 1483]
Longitud original: 5 palabras

3. FRASE CON PADDING (X_train_pad):
[ 105   30    5   27 1483    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0]
Longitud final: 36 (rellenada con ceros)


### Parámetros

In [43]:
# Tus parámetros definidos
vocab_size = len(word_index) + 1
embedding_dim = 32

# Diccionario de configuración para producción
config_produccion = {
    "vocab_size": int(vocab_size),
    "embedding_dim": int(embedding_dim),
    "max_len": int(max_len)
}

# Guardamos en un archivo JSON
with open('config_modelo_rnn.json', 'w') as f:
    json.dump(config_produccion, f, indent=4)

print("✅ Parámetros de arquitectura guardados en 'config_modelo.json'")

✅ Parámetros de arquitectura guardados en 'config_modelo.json'


In [44]:
# Forzamos que ambos sean arreglos de tipo float32 o int32
X_train_pad = np.array(X_train_pad).astype(np.int32)
y_train = np.array(y_train).astype(np.int32)

X_test_pad = np.array(X_test_pad).astype(np.int32)
y_test = np.array(y_test).astype(np.int32)

X_val_pad = np.array(X_val_pad).astype(np.int32)
y_val = np.array(y_val).astype(np.int32)

In [45]:
# Asegúrate de importar tu módulo primero
import Funciones_PNL as f
import importlib
importlib.reload(f) # Si haces cambios, esto recarga el archivo
loss_function = 'sparse_categorical_crossentropy'
# 2. Definir qué arquitecturas quieres comparar
mis_arquitecturas = ['SimpleRNN', 'LSTM', 'GRU']
num_classes = 11

resultados = f.ejecutar_benchmarking(
    X_train=X_train_pad,
    y_train=y_train,
    X_val=X_val_pad,
    y_val=y_val,
    X_test=X_test_pad,
    y_test=y_test,
    num_classes=num_classes,
    loss_fn=loss_function,
    vocab_size=vocab_size,
    max_len=max_len,
    texto_nombremodelo="proyecto_nlp",
    epochs=20,
    max_trials=15 # Aumenta esto para probar más combinaciones
)

print(resultados)

Trial 10 Complete [00h 00m 55s]
val_accuracy: 0.5105619430541992

Best val_accuracy So Far: 0.7674996256828308
Total elapsed time: 00h 11m 21s

Search: Running Trial #11

Value             |Best Value So Far |Hyperparameter
96                |224               |units
0.3               |0.2               |dropout
0.0005            |0.0005            |lr
32                |128               |embedding_dim

Epoch 1/20
1057/1057 ━━━━━━━━━━━━━━━━━━━━ 13s 9ms/step - accuracy: 0.3412 - loss: 2.2877 - val_accuracy: 0.5843 - val_loss: 1.6062
Epoch 2/20
1057/1057 ━━━━━━━━━━━━━━━━━━━━ 16s 6ms/step - accuracy: 0.6425 - loss: 1.7525 - val_accuracy: 0.6147 - val_loss: 1.3502
Epoch 3/20
  56/1057 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.6540 - loss: 2.3805

KeyboardInterrupt: 

In [ ]:
import shutil
shutil.rmtree('logs', ignore_errors=True)
print("Carpeta 'logs' eliminada correctamente.")

In [ ]:
import inspect
print(inspect.signature(f.ejecutar_benchmarking))

In [ ]:
gha562789

In [ ]:
import Funciones_PNL as f  # <--- Esto es un módulo
import importlib
importlib.reload(f)     # <--- Esto sí funciona
loss_function = 'sparse_categorical_crossentropy'
# 2. Definir qué arquitecturas quieres comparar
mis_arquitecturas = ['SimpleRNN', 'LSTM', 'GRU']
num_classes = 11

resultados = f.ejecutar_benchmarking(
    X_train=X_train_pad,
    y_train=y_train,
    X_val=X_test_pad,
    y_val=y_test,
    num_classes=num_classes,
    loss_fn=loss_function,
    vocab_size=vocab_size,
    embedding_dim=embedding_dim,
    max_len=max_len,
    texto_nombremodelo="modelo_simple",
    arquitecturas=["SimpleRNN", "LSTM", "GRU"],
    epochs=10
)
print(resultados)

## Modelo Elman

In [ ]:
import Funciones_PNL as f  # <--- Esto es un módulo
import importlib
importlib.reload(f)     # <--- Esto sí funciona

In [ ]:
loss_function = 'sparse_categorical_crossentropy'
# 2. Definir qué arquitecturas quieres comparar
mis_arquitecturas = ['SimpleRNN', 'LSTM', 'GRU']
num_classes = 11

In [ ]:
import Funciones_PNL as f  # <--- Esto es un módulo
import importlib
importlib.reload(f)     # <--- Esto sí funciona
df_comparativa = f.ejecutar_benchmarking(
    X_train=X_train_pad,
    y_train=y_train,
    X_val=X_test_pad,
    y_val=y_test,
    num_classes=num_classes,
    loss_fn=loss_function,
    vocab_size=vocab_size,
    embedding_dim=embedding_dim,
    max_len=max_len,
    arquitecturas=mis_arquitecturas
)

# 4. (Opcional) Guardar el reporte maestro a Excel o CSV para tu reporte de resultados
df_comparativa.to_csv("reporte_final_benchmarking.csv", index=False)
print("\n>>> Reporte guardado como 'reporte_final_benchmarking.csv'")

In [ ]:
kljhajgsjh1562782

In [ ]:
# --- ENTRENAMIENTO DEL MODELO SimpleRNN ---
print(">>> Entrenando SimpleRNN...")


# 1. Calcular los pesos automáticamente basados en tu 'y_train'
class_weights = class_weight.compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)

# 2. Convertir a diccionario
class_weight_dict = dict(enumerate(class_weights))

# Almacenamos el proceso en 'history_rnn' para poder graficar el aprendizaje después
history_rnn = model_rnn.fit(
    X_train_pad,        # Tensores de entrenamiento (960 muestras, max_len columnas)
    y_train,            # Etiquetas reales (0 o 1)
    class_weight=class_weight_dict,
    epochs=40,          # El modelo verá el dataset completo 10 veces
    batch_size=64,      # Los pesos se actualizan cada 32 ejemplos procesados
    validation_data=(X_test_pad, y_test), # Evaluación con datos no vistos en cada época
    verbose=1           # Muestra la barra de progreso y métricas en tiempo real
)

In [ ]:
# 1. Construcción del Modelo LSTM
model_lstm = Sequential([
    # Capa de entrada con la longitud máxima calculada
    Input(shape=(max_len,)),

    # Capa Embedding: mask_zero=True es vital para que la LSTM ignore el padding
    Embedding(input_dim=vocab_size, output_dim=embedding_dim, mask_zero=True),

    # Capa LSTM: Ya no se pone la función de activación
    # 32 unidades para capturar patrones más complejos que la RNN simple.
    LSTM(128, recurrent_dropout=0.3),

    Dense(11, activation='softmax')
])

# 2. Configuración del Entrenamiento
# Usamos una tasa de aprendizaje ligeramente menor para mayor estabilidad
model_lstm.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Resumen del modelo
model_lstm.summary()

In [ ]:
print("\n" + "="*50 + "\n")

# --- ENTRENAMIENTO DEL MODELO LSTM ---
print(">>> Entrenando LSTM...")

# La LSTM suele requerir más capacidad de cómputo por sus compuertas internas
history_lstm = model_lstm.fit(
    X_train_pad,        # Usamos los mismos datos para que la comparación sea justa
    y_train,
    epochs=10,
    batch_size=64,
    validation_data=(X_test_pad, y_test),
    verbose=1           # Monitoreamos 'loss', 'accuracy', 'val_loss' y 'val_accuracy'
)

In [ ]:
from tensorflow.keras.layers import Embedding, LSTM, GRU, Dense, Input # Capas específicas para PLN y redes neuronales

# 1. Construcción del Modelo LSTM
model_gru = Sequential([
    # Capa de entrada con la longitud máxima calculada
    Input(shape=(max_len,)),

    # Capa Embedding: mask_zero=True es vital para que la LSTM ignore el padding
    Embedding(input_dim=vocab_size, output_dim=embedding_dim, mask_zero=True),

    # Capa LSTM: Ya no se pone la función de activación
    # 32 unidades para capturar patrones más complejos que la RNN simple.
    GRU(32,recurrent_dropout=0.2),

    Dense(11, activation='softmax')
])

# 2. Configuración del Entrenamiento
# Usamos una tasa de aprendizaje ligeramente menor para mayor estabilidad
model_gru.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Resumen del modelo
model_lstm.summary()

In [ ]:
from sklearn.utils import class_weight
import numpy as np

# 1. Calculamos los pesos de clase
# 'y_train' debe ser una lista o array con los valores numéricos (0 a 10)
class_weights = class_weight.compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)

# 2. Convertimos a diccionario (Keras requiere el formato {clase: peso})
class_weight_dict = dict(enumerate(class_weights))

print("Pesos asignados por clase:")
for clase, peso in class_weight_dict.items():
    print(f"Clase {clase}: {peso:.2f}")

print("\n" + "="*50 + "\n")

# --- ENTRENAMIENTO DEL MODELO CON BALANCEO ---
print(">>> Entrenando modelo con Class Weights para balanceo...")

history_gru = model_gru.fit(
    X_train_pad,
    y_train,
    epochs=10,
    batch_size=32,
    validation_data=(X_test_pad, y_test),
    class_weight=class_weight_dict, # AQUÍ APLICAS EL BALANCEO
    verbose=1
)